In [1]:
import os
import json
import random
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

In [2]:
load_dotenv(override=True)

API_KEY = os.getenv("GAPGPT_API_KEY")
BASE_URL = "https://api.gapgpt.app/v1"
MODEL = "gpt-4o"
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [3]:
# 1) System Prompt
system_message = (
    "You are a helpful assistant for an Airline called FlightAI.\n\n"
    "When the user wants to book a flight, follow these steps:\n"
    "1. Ask for the source city.\n"
    "2. Ask for the destination city (must be different from source).\n"
    "3. Call the function 'check_flight_availability' with the user's destination.\n"
    "   - If it returns an empty list, say: 'No flights to that city'.\n"
    "   - If it returns flights, list them EXACTLY, in a numbered list, showing airline, time, price, and duration.\n"
    "4. Wait for the user to pick one flight option by number.\n"
    "5. Then ask for passenger first name, last name, and age.\n"
    "6. Finally call 'book_flight' to confirm and show the user the real seat number and booking details.\n\n"
    "You also have a tool 'generate_report' which summarizes ALL booked tickets in a single file.\n\n"
    "IMPORTANT:\n"
    "- Always call 'check_flight_availability' if user mentions a new destination.\n"
    "- Do not invent flights or seat numbers. Use what the function calls return.\n"
    "- Source and destination cannot be the same.\n"
    "- Every time a flight is booked, produce a new ticket file named firstName_lastName_bookingNumber.txt.\n"
    "- If a city is not in flight_availability, say 'No flights found for that city'.\n"
    "If the user wants all tickets summarized, call 'generate_report' with no arguments (the function has none).\n"
    "If you don't know something, say so.\n"
    "Keep answers short and courteous.\n"
)

In [4]:
# 2) Flight Availability with Price & Duration
flight_availability = {
    "london": [
        {
            "airline": "AirlinesAI",
            "time": "10:00 AM",
            "price": "$799",
            "duration": "8 hours"
        },
        {
            "airline": "IndianAirlinesAI",
            "time": "3:00 PM",
            "price": "$899",
            "duration": "8 hours"
        },
        {
            "airline": "AmericanAirlinesAI",
            "time": "8:00 PM",
            "price": "$999",
            "duration": "8 hours"
        },
    ],
    "paris": [
        {
            "airline": "EuropeanAirlinesAI",
            "time": "11:00 AM",
            "price": "$399",
            "duration": "7 hours"
        },
        {
            "airline": "BudgetAirlines",
            "time": "6:00 PM",
            "price": "$2399",
            "duration": "7 hours"
        },
    ],
    "tokyo": [
        {
            "airline": "TokyoAirlinesAI",
            "time": "12:00 PM",
            "price": "$4000",
            "duration": "5 hours"
        },
        {
            "airline": "FastFly",
            "time": "7:00 PM",
            "price": "$1400",
            "duration": "5 hours"
        },
    ],
    "berlin": [
        {
            "airline": "BerlinAirlinesAI",
            "time": "9:00 AM",
            "price": "$499",
            "duration": "6 hours"
        },
        {
            "airline": "AmericanAirlinesAI",
            "time": "4:00 PM",
            "price": "$899",
            "duration": "6 hours"
        },
    ],
    "nagpur": [
        {
            "airline": "IndianAirlinesAI",
            "time": "8:00 AM",
            "price": "$1000",
            "duration": "10 hours"
        },
        {
            "airline": "JetAirlines",
            "time": "2:00 PM",
            "price": "$1500",
            "duration": "10 hours"
        },
        {
            "airline": "AirlinesAI",
            "time": "10:00 PM",
            "price": "$800",
            "duration": "10 hours"
        },
    ],
}


In [5]:
# A global list of flight bookings
flight_bookings = []

In [ ]:
# 3) Helper Functions
def generate_seed_numbers(seed_value):
    random.seed(seed_value)
    return [
        f"{random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')}{random.randint(1, 99):02}"
        for _ in range(5)
    ]

def check_flight_availability(destination_city: str):
    """
    Return the flight for a given city from flight_availability(),
    If no flight available returns an empty list.
    """

    print(f"[TOOL] check_flight_availability({destination_city})")
    city = destination_city.lower()
    return flight_availability.get(city, [])

def generate_ticket_file(booking_dict, booking_number):

    fname = booking_dict["first_name"].replace(" ", "_")
    lname = booking_dict["last_name"].replace(" ", "_")
    filename = f"{fname}_{lname}_{booking_number}.txt"

    content = (
        "Flight Ticket\n"
        "=============\n"
        f"Booking #   : {booking_number}\n"
        f"Passenger   : {booking_dict['first_name']} {booking_dict['last_name']}, Age {booking_dict['age']}\n"
        f"Source      : {booking_dict['source']}\n"
        f"Destination : {booking_dict['destination']}\n"
        f"Airline     : {booking_dict['airline']}\n"
        f"Departure   : {booking_dict['time']}\n"
        f"Price       : {booking_dict['price']}\n"
        f"Duration    : {booking_dict['duration']}\n"
        f"Seat Number : {booking_dict['seat']}\n"
    )

    with open(filename, "w") as f:
       f.write(content)

    print(f"[TOOL] Ticket file generated => {filename}")
    return filename

def book_flight(source, destination, option_index, fname, lname, age):
    """
    Book a flight using an option index for the chosen city.
    - source != destination
    - index is 1-based => we do pick = idx - 1
    - create new booking record, seat assignment, & ticket file
    """
    print(f"[TOOL] book_flight({source=}, {destination=}, {option_index=})")

    if source.lower() == destination.lower():
        return "Souce annd Destination must not be same."
    
    # Convert option index from string to integer
    try:
        idx = int(option_index)
    except ValueError:
        return "Error: Option index is not a valid integer."
    
    flights = check_flight_availability(destination)
    if not flights:
        return f"Error: No flights found for {destination.title()}."
    
    pick = idx - 1
    if pick == 0 or pick >= len(flights):
        return f"Error: Invalid flight option #{idx} for {destination.tilte()}"
    
    chosen_flight = flights[pick]
    airline = chosen_flight["airline"]
    dep_time = chosen_flight["time"]
    price = chosen_flight["price"]
    duration = chosen_flight["duration"]

    # Generate seat
    seat_list = generate_seed_numbers(hash(destination + airline + str(len(flight_bookings))))
    chosen_seat = seat_list[0]

    new_booking = {
        "source":      source.title(),
        "destination": destination.title(),
        "airline":     airline,
        "time":        dep_time,
        "price":       price,
        "duration":    duration,
        "seat":        chosen_seat,
        "first_name":  fname.title(),
        "last_name":   lname.title(),
        "age":         age,
    }
    flight_bookings.append(new_booking)

    booking_number = len(flight_bookings)
    ticket_filename = generate_ticket_file(new_booking, booking_number)

    confirmation = (
        f"Booking #{booking_number} confirmed for {first_name.title()} {last_name.title()}. "
        f"Flight from {source.title()} to {destination.title()} on {airline} at {dep_time}. "
        f"Ticket saved to {ticket_filename}."
    )
    print(f"[TOOL] {confirmation}")
    return confirmation

def generate_report():
    """
    Summarize all tickets in a single file called summary_report.txt
    """
    print(f"[TOOL] generate_report() called.")
    report_content = "Flight Booking Summary Report\n"
    report_content += "=============================\n"

    if not flight_bookings:
        report_content += "No booking found. \n"
    else:
        for i, booking in enumerate(flight_bookings, start=1):
            report_content += (
                f"Booking #   : {i}\n"
                f"Passenger   : {booking['first_name']} {booking['last_name']}, Age {booking['age']}\n"
                f"Source      : {booking['source']}\n"
                f"Destination : {booking['destination']}\n"
                f"Airline     : {booking['airline']}\n"
                f"Departure   : {booking['time']}\n"
                f"Price       : {booking['price']}\n"
                f"Duration    : {booking['duration']}\n"
                f"Seat Number : {booking['seat']}\n"
                "-------------------------\n"
            )

            filename = "summary_report.txt"
            with open(filename, "w") as f:
                f.write(report_content)
            
            msg = f"Summary report generated => {filename}"
            print(f"[TOOL] {msg}")
            return msg

In [ ]:
# 4) Tools JSON Schemas
price_function = {
    "name": "get_ticket_price",
    "description": "Get the ticket price of a flight to the requested city (destination city)",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city customer wants to travel to."
            },
        },
        "required": ["destination_city"],
    },
}

availability_function = {
    "name": "check_flight_availability",
    "description": "Check if a any airline's flight to the desire city is available or not.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "Name of the city customer wants to travel to.",
            },
        },
        "required": ["destination_citys"],
    },
}

book_function = {
    "name": "book_flight_function",
    "description": (
        "Get the travel and customer information, check if any airlines available and based on the information, "
        "saves and return the booking result."
        "Generates a unique ticket file named firstname_lastname_bookingNumber.txt each time."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "source": {
                "type": "string",
                "description": "Source city in which customer travle's starts from.",
            },
            "destination": {
                "type": "string",
                "description": "The city customer wants to travel to."
            },
            "option_index": {
                "type": "integer",
                "description": "Index number of airline which customer selects. If there were multiple airline's option.",
            },
            "fname": {
                "type": "string",
                "description": "First name of the customer.",
            },
            "lname": {
                "type": "string",
                "description": "Last name of the customer.",
            },
            "age": {
                "type": "integer",
                "description": "Age of the customer.",
            },
        },
        "required": ["source", "destination_city", "fname", "lname", "age"],
    },
}

report_function = {
    "name": "generate_report",
    "description": "Generate a summary report of ALL tickets in a single file called summary_report.txt.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
    },
}

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": availability_function},
    {"type": "function", "function": book_function},
    {"type": "function", "function": report_function},
]

In [10]:
# 5) Handle Tool Calls
def handle_tool_call(message):
    """
    LLM could request to call a function. We recieve the request, parse it in JSON,
    And run the python function. Result will be a 'too' message.
    """
    tool_call = message.tool_calls[0]
    fn_name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    if fn_name == "get_ticket_price":
        city = args.get("destination_city")
        flights = check_flight_availability(city)

        if not flights:
            response_content = {"destination_city": city, "price": "No flight found."}
        else:
            price = flights[0]["price"]
            response_content = {"destination_city": city, "price": price}
    elif fn_name == "check_flight_availability":
        city = args.get("destination_city")
        flights = check_flight_availability(city)
        response_content = {"destination_city": city, "availability": flights}
    elif fn_name == "book_function":
        src = args.get("source")
        dest = args.get("destination")
        idx = args.get("option_index")
        fname = args.get("fname")
        lname = args.get("lname")
        age = args.get("age")

        confirmation = book_flight(src, dest, idx, fname, lname, age)
        response_content = {
            "source": src,
            "destination": dest,
            "option_index": idx,
            "fname": fname,
            "lname": lname,
            "age": age,
            "confirmation": confirmation,
        }
    elif fn_name == "generate_report":
        result = generate_report()
        response_content = {"report": result}
    else:
        response_content = {"error": f"Unknown tool: {fn_name}"}

    return {
        "role": "tool",
        "content": json.dumps(response_content),
        "tool_call_id": tool_call.id,
    }, args
    

In [11]:
# 6) Main Chat Function
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )

        while response.choices[0].finish_reason == "tool_calls":
            msg = response.choices[0].message
            print(f"[INFO] tool call requested: {msg.tool_calls[0]}")
            tool_response, tool_args = handle_tool_call(msg)
            print(f"[INFO] tool call response: {tool_response}")

            messages.append(msg)
            messages.append(tool_response)

            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
            )
            
        return response.choices[0].messagge.content
    
    except Exception as e:
        print(f"[ERROR] {e}")
        return "I'm sorry, something went wrong while processing your request!"

In [14]:
# 7) Launch Gradio
gr.ChatInterface(fn=chat, type="messages").launch()

TypeError: ChatInterface.__init__() got an unexpected keyword argument 'type'

In [100]:
# 8) Rewrite check_flight_availability function with sqlite
import sqlite3

DB = "flights.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS flights (city TEXT PRIMARY KEY, airline TEXT," \
    "time TEXT, price REAL, duration TEXT)")
    conn.commit()

In [85]:
def check_flight_availability(destination_city: str):
    """
    Return the flight for a given city from flights Database,
    If no flight available returns an empty list.
    """

    print(f"[TOOL] check_flight_availability({destination_city})")
    city = destination_city.lower()
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT airline, time, price, duration FROM flights WHERE city = ?", (city,))
        result = cursor.fetchmany()
    return result if result else []

In [97]:
def set_flight(**kwargs):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        for city, flight_details in kwargs.items():
            for item in flight_details:
                cursor.execute("INSERT INTO flights (city, airline, time, price, duration) VALUES (?, ?, ?, ?, ?)",
                (city.lower(), item["airline"], item["time"], item["price"], item["duration"]))
        conn.commit()

In [98]:
set_flight(**flight_availability)

IntegrityError: UNIQUE constraint failed: flights.city

In [87]:
check_flight_availability("paris")

[TOOL] check_flight_availability(paris)


[('BudgetAirlines', '6:00 PM', '$2399', '7 hours')]